### Trabalho Fase 2 do Curso de Pos-Graduacao FIAP IA para Devs
#### Parte 10 - Integracao com LLMs usando ChatGPT

Fonte de dados escolhida: DATASUS/SISCAN  
Tipo de dados de origem nesta etapa: resultados dos modelos e payloads estruturados  

Arquivos utilizados quando disponiveis:

```text
bases/teste/x.parquet
bases/teste/y.parquet
resultados/parte_10_integracao_llms/
```

---

## Objetivo da Parte 10

Integrar uma LLM pre-treinada ao fluxo de modelagem para transformar predicoes individuais e metricas agregadas em explicacoes compreensiveis, cautelosas e auditaveis.

A LLM nao substitui o classificador e nao emite diagnostico. Neste notebook, ela atua como camada de interpretacao dos resultados estatisticos gerados nas partes anteriores.

#### Item 1 - Configuracao da integracao

A integracao usa a API da OpenAI por variaveis de ambiente, sem gravar chave no notebook.

Antes de executar com ChatGPT real, configurar:

```bash
export OPENAI_API_KEY="sua-chave"
export OPENAI_MODEL="gpt-5.5"
```

Se `OPENAI_API_KEY` nao estiver configurada, o notebook usa um cliente local demonstrativo para permitir validar os prompts, o formato de saida e a avaliacao automatica.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import os
import re
from typing import Any

try:
    import pandas as pd
except ImportError as exc:
    raise ImportError("Instale as dependencias com: pip install -r requirements.txt") from exc

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value):
        print(value)

    class Markdown(str):
        pass

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "resultados" / "parte_10_integracao_llms"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")
HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))

pd.set_option("display.max_colwidth", 140)

print(f"Modelo LLM configurado: {OPENAI_MODEL}")
print(f"Chave OpenAI configurada: {HAS_OPENAI_KEY}")
print(f"Diretorio de saida: {OUTPUT_DIR}")

#### Item 2 - Prompts de seguranca e interpretacao

Os prompts abaixo delimitam o papel da LLM. A resposta deve explicar resultados estatisticos, comunicar incerteza e evitar diagnostico definitivo, prescricao ou conduta clinica obrigatoria.

In [ ]:
SYSTEM_PROMPT = """
Voce e um assistente de apoio a interpretacao de modelos de triagem em saude.
Explique resultados de machine learning em linguagem clara para medicos e gestores.
Nao forneca diagnostico definitivo, prescricao ou conduta clinica obrigatoria.
Deixe claro quando a informacao vier de um modelo estatistico e quando houver incerteza.
Priorize recall, precision, F1 e balanced accuracy em bases desbalanceadas.
Nao invente sintomas, achados clinicos, antecedentes ou variaveis que nao estejam no payload.
Responda em portugues do Brasil, com tom tecnico, objetivo e cauteloso.
""".strip()

INDIVIDUAL_SECTIONS = [
    "Resumo da predicao",
    "Fatores observados",
    "Interpretacao da incerteza",
    "Acao operacional sugerida",
    "Limitacoes",
]

AGGREGATE_SECTIONS = [
    "Resumo executivo",
    "Leitura das metricas",
    "Impacto clinico-operacional",
    "Comparacao entre modelos",
    "Recomendacoes para validacao",
]

PROHIBITED_TERMS = [
    "diagnostico confirmado",
    "cancer confirmado",
    "paciente tem cancer",
    "paciente nao tem cancer",
    "tratamento indicado",
    "iniciar tratamento",
]

In [ ]:
def to_builtin(value: Any) -> Any:
    """Converte valores do pandas/numpy para tipos serializaveis em JSON."""
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def sanitize_patient_payload(record: dict[str, Any], max_features: int = 12) -> dict[str, Any]:
    """Remove campos potencialmente identificadores e limita o payload enviado a LLM."""
    blocked_patterns = [
        "nome", "cpf", "cns", "cartao", "endereco", "telefone", "email",
        "municipio", "cep", "logradouro", "bairro", "mae", "nascimento",
    ]
    sanitized = {}
    for key, value in record.items():
        key_lower = str(key).lower()
        if any(pattern in key_lower for pattern in blocked_patterns):
            continue
        if value is None or pd.isna(value):
            continue
        sanitized[str(key)] = to_builtin(value)
        if len(sanitized) >= max_features:
            break
    return sanitized


def build_individual_explanation_prompt(payload: dict[str, Any]) -> str:
    payload_json = json.dumps(payload, ensure_ascii=False, indent=2)
    sections = "\n".join(f"- {section}" for section in INDIVIDUAL_SECTIONS)
    return f"""
Contexto:
O modelo abaixo foi treinado para triagem de risco em mamografias do SISCAN.
A classe positiva indica maior probabilidade estatistica de cancer de mama provavel.
O modelo nao substitui avaliacao medica.

Payload estruturado, sem identificacao pessoal:
{payload_json}

Tarefa:
1. Explique em linguagem natural o que a predicao significa.
2. Destaque apenas fatores estruturados presentes no payload.
3. Explique a incerteza usando as metricas do modelo.
4. Sugira uma acao operacional de triagem, sem prescrever diagnostico ou tratamento.
5. Liste limitacoes do modelo para este caso.

Responda exatamente com estas secoes:
{sections}
""".strip()


def build_model_summary_prompt(payload: dict[str, Any]) -> str:
    payload_json = json.dumps(payload, ensure_ascii=False, indent=2)
    sections = "\n".join(f"- {section}" for section in AGGREGATE_SECTIONS)
    return f"""
Contexto:
Estamos avaliando modelos de triagem para uma base desbalanceada do SISCAN.
A classe positiva e rara, portanto accuracy isolada nao deve ser usada como criterio principal.

Metricas e resultados estruturados:
{payload_json}

Tarefa:
1. Resuma o desempenho para um publico medico e gestor.
2. Explique o trade-off entre recall e precision.
3. Indique impacto operacional de falsos positivos e falsos negativos.
4. Compare baseline e modelos otimizados quando os dados estiverem disponiveis.
5. Gere recomendacoes acionaveis para validacao futura.

Responda exatamente com estas secoes:
{sections}
""".strip()


def validate_llm_response(response: str, required_sections: list[str], numbers: list[float] | None = None) -> dict[str, Any]:
    response_lower = response.lower()
    missing_sections = [section for section in required_sections if section.lower() not in response_lower]
    prohibited_found = [term for term in PROHIBITED_TERMS if term in response_lower]
    has_uncertainty = any(term in response_lower for term in ["incerteza", "limitacao", "limitacoes", "modelo estatistico", "triagem"])

    missing_numbers = []
    for number in numbers or []:
        text_number = f"{number:.2f}".replace(".", ",")
        alt_number = f"{number:.2f}"
        if text_number not in response and alt_number not in response:
            missing_numbers.append(number)

    passed = not missing_sections and not prohibited_found and has_uncertainty and not missing_numbers and len(response) <= 7000
    return {
        "aprovado": passed,
        "secoes_ausentes": missing_sections,
        "termos_proibidos": prohibited_found,
        "menciona_incerteza_ou_triagem": has_uncertainty,
        "numeros_ausentes": missing_numbers,
        "tamanho_caracteres": len(response),
    }

#### Item 3 - Cliente LLM

A classe `OpenAIResponsesClient` encapsula a chamada ao ChatGPT pela API. A interface `generate` permite trocar o provedor no futuro sem alterar a preparacao dos prompts.

In [ ]:
@dataclass
class LLMClient:
    def generate(self, system_prompt: str, user_prompt: str) -> str:
        raise NotImplementedError


@dataclass
class OpenAIResponsesClient(LLMClient):
    model: str = OPENAI_MODEL
    max_output_tokens: int = 1400

    def __post_init__(self):
        from openai import OpenAI
        self.client = OpenAI()

    def generate(self, system_prompt: str, user_prompt: str) -> str:
        response = self.client.responses.create(
            model=self.model,
            instructions=system_prompt,
            input=user_prompt,
            max_output_tokens=self.max_output_tokens,
        )
        return response.output_text


@dataclass
class DemoLLMClient(LLMClient):
    """Cliente local para demonstrar o fluxo quando a chave da OpenAI nao esta configurada."""

    def generate(self, system_prompt: str, user_prompt: str) -> str:
        recall_values = [float(value) for value in re.findall(r'"recall":\s*([0-9.]+)', user_prompt)]
        precision_values = [float(value) for value in re.findall(r'"precision":\s*([0-9.]+)', user_prompt)]
        f1_values = [float(value) for value in re.findall(r'"f1":\s*([0-9.]+)', user_prompt)]
        recall_text = f"{max(recall_values):.2f}" if recall_values else "nao informado"
        precision_text = f"{max(precision_values):.2f}" if precision_values else "nao informado"
        f1_text = f"{max(f1_values):.2f}" if f1_values else "nao informado"

        if "Resumo da predicao" in user_prompt:
            return f"""
Resumo da predicao:
O classificador indicou maior risco estatistico para a classe positiva, mas isso deve ser interpretado apenas como apoio a triagem, nao como diagnostico clinico.

Fatores observados:
A interpretacao deve considerar somente as variaveis estruturadas enviadas no payload, como idade, historico de mamografia anterior e demais campos codificados disponiveis.

Interpretacao da incerteza:
Como a base e desbalanceada, precision, recall, F1 e balanced accuracy sao mais informativas que accuracy isolada. Neste payload, o recall informado e {recall_text} e o F1 informado e {f1_text}. Um recall maior reduz falsos negativos, mas pode aumentar falsos positivos.

Acao operacional sugerida:
Priorizar revisao operacional do caso conforme protocolo local de triagem e disponibilidade de avaliacao profissional, sem concluir presenca ou ausencia de cancer.

Limitacoes:
A resposta depende das metricas globais do modelo e das variaveis estruturadas enviadas. O modelo nao incorpora exame fisico, imagem, laudo completo ou julgamento medico.
""".strip()

        return f"""
Resumo executivo:
Os modelos devem ser avaliados como ferramentas de apoio a triagem em uma base desbalanceada, com atencao prioritaria a recall, precision, F1 e balanced accuracy.

Leitura das metricas:
Accuracy isolada pode ser enganosa quando a classe positiva e rara. Nos dados enviados, o maior recall observado e {recall_text} e a maior precision observada e {precision_text}. Recall indica a capacidade de capturar casos positivos e precision indica a proporcao de alertas positivos que realmente correspondem a classe positiva.

Impacto clinico-operacional:
Falsos negativos podem atrasar priorizacao de casos relevantes. Falsos positivos aumentam revisoes e carga operacional, mas podem ser aceitaveis em estrategias de triagem quando controlados.

Comparacao entre modelos:
A escolha deve considerar o equilibrio entre recall e precision, alem da estabilidade em validacao e da interpretabilidade operacional.

Recomendacoes para validacao:
Validar em dados futuros, revisar erros por subgrupos, calibrar limiares e submeter interpretacoes a revisao humana antes de qualquer uso real.
""".strip()


llm_client: LLMClient
if HAS_OPENAI_KEY:
    llm_client = OpenAIResponsesClient()
else:
    llm_client = DemoLLMClient()

print(type(llm_client).__name__)

#### Item 4 - Consolidacao de metricas dos modelos

Como os notebooks anteriores documentam resultados dentro dos proprios notebooks, esta secao cria um payload agregado com os valores baseline ja registrados. Ao reexecutar as partes 7, 8 e 9, os valores otimizados podem ser substituidos pelos resultados finais recalculados.

In [ ]:
model_metrics = pd.DataFrame([
    {
        "modelo": "SVM baseline Parte 4",
        "tipo": "baseline",
        "accuracy": 0.6811,
        "balanced_accuracy": 0.6066,
        "precision": 0.0693,
        "recall": 0.5251,
        "f1": 0.1225,
        "roc_auc": 0.6492,
    },
    {
        "modelo": "Regressao Logistica baseline Parte 6",
        "tipo": "baseline",
        "accuracy": 0.6784,
        "balanced_accuracy": 0.5924,
        "precision": 0.0657,
        "recall": 0.4985,
        "f1": 0.1161,
        "roc_auc": 0.6348,
    },
    {
        "modelo": "Random Forest baseline Parte 5",
        "tipo": "baseline",
        "accuracy": 0.7971,
        "balanced_accuracy": 0.5924,
        "precision": 0.0815,
        "recall": 0.3687,
        "f1": 0.1335,
        "roc_auc": 0.6248,
    },
])

model_metrics

#### Item 5 - Payload individual sem identificacao pessoal

O exemplo abaixo tenta usar um registro real do conjunto de teste. Apenas variaveis estruturadas nao identificadoras sao mantidas. Se a base nao estiver disponivel, o notebook usa um exemplo sintetico.

In [ ]:
def load_example_patient_payload() -> dict[str, Any]:
    x_path = BASE_DIR / "bases" / "teste" / "x.parquet"
    y_path = BASE_DIR / "bases" / "teste" / "y.parquet"

    if x_path.exists():
        x_test = pd.read_parquet(x_path)
        row = x_test.iloc[0].to_dict()
        variables = sanitize_patient_payload(row)
        target_real = None
        if y_path.exists():
            y_test = pd.read_parquet(y_path)
            target_real = to_builtin(y_test.iloc[0, 0])
        return {
            "origem": "bases/teste/x.parquet",
            "target_real_para_auditoria": target_real,
            "variaveis_relevantes": variables,
        }

    return {
        "origem": "exemplo sintetico",
        "target_real_para_auditoria": None,
        "variaveis_relevantes": {
            "CO_IDADE_PACIENTE_NUM": 63,
            "CO_TEMPO_MAMO_ANTERIOR_NUM": 3,
            "TP_MAMOGRAFIA_RASTREAMENT": "rastreamento",
            "CO_RACA_COR": "informada",
        },
    }


best_baseline = model_metrics.sort_values(["f1", "balanced_accuracy"], ascending=False).iloc[0].to_dict()
patient_context = load_example_patient_payload()

individual_payload = {
    "modelo": best_baseline["modelo"],
    "classe_prevista": 1,
    "score_risco": 0.68,
    "limiar_classificacao": 0.50,
    "observacao": "Score demonstrativo; substituir pelo score real do modelo escolhido quando a parte de predicao estiver reexecutada.",
    "metricas_modelo": {
        "recall": round(float(best_baseline["recall"]), 4),
        "precision": round(float(best_baseline["precision"]), 4),
        "f1": round(float(best_baseline["f1"]), 4),
        "balanced_accuracy": round(float(best_baseline["balanced_accuracy"]), 4),
    },
    "contexto_registro": patient_context,
}

display(individual_payload)

#### Item 6 - Geracao de explicacao individual

Esta chamada envia somente o prompt estruturado e o payload sanitizado. A resposta e salva em Markdown para auditoria.

In [ ]:
individual_prompt = build_individual_explanation_prompt(individual_payload)
individual_response = llm_client.generate(SYSTEM_PROMPT, individual_prompt)

individual_validation = validate_llm_response(
    individual_response,
    INDIVIDUAL_SECTIONS,
    numbers=[individual_payload["metricas_modelo"]["recall"], individual_payload["metricas_modelo"]["f1"]],
)

(OUTPUT_DIR / "prompt_individual.txt").write_text(individual_prompt, encoding="utf-8")
(OUTPUT_DIR / "resposta_individual.md").write_text(individual_response, encoding="utf-8")
(OUTPUT_DIR / "validacao_individual.json").write_text(json.dumps(individual_validation, ensure_ascii=False, indent=2), encoding="utf-8")

display(Markdown(individual_response))
display(individual_validation)

#### Item 7 - Geracao de interpretacao agregada

A interpretacao agregada transforma as metricas em leitura clinico-operacional, com foco no desbalanceamento da base e no trade-off entre falsos negativos e falsos positivos.

In [ ]:
aggregate_payload = {
    "objetivo_aplicacao": "apoio a triagem e priorizacao em mamografias do SISCAN",
    "classe_positiva": "maior probabilidade estatistica de cancer de mama provavel",
    "observacoes": [
        "A base e desbalanceada; accuracy isolada nao deve orientar a escolha do modelo.",
        "Recall maior reduz falsos negativos, mas pode aumentar carga operacional por falsos positivos.",
        "Os valores otimizados por AG devem ser atualizados quando as partes 7, 8 e 9 forem reexecutadas.",
    ],
    "metricas_modelos": model_metrics.to_dict(orient="records"),
}

aggregate_prompt = build_model_summary_prompt(aggregate_payload)
aggregate_response = llm_client.generate(SYSTEM_PROMPT, aggregate_prompt)

aggregate_validation = validate_llm_response(
    aggregate_response,
    AGGREGATE_SECTIONS,
    numbers=[round(float(model_metrics["recall"].max()), 2), round(float(model_metrics["precision"].max()), 2)],
)

(OUTPUT_DIR / "prompt_agregado.txt").write_text(aggregate_prompt, encoding="utf-8")
(OUTPUT_DIR / "resposta_agregada.md").write_text(aggregate_response, encoding="utf-8")
(OUTPUT_DIR / "validacao_agregada.json").write_text(json.dumps(aggregate_validation, ensure_ascii=False, indent=2), encoding="utf-8")

display(Markdown(aggregate_response))
display(aggregate_validation)

#### Item 8 - Tabela final de auditoria

A tabela abaixo registra prompt, modelo LLM, tipo de interpretacao e resultado da validacao automatica. Em um uso real, esta tabela deve ser complementada por revisao humana, preferencialmente com especialista clinico.

In [ ]:
audit_rows = [
    {
        "tipo_interpretacao": "individual",
        "llm_client": type(llm_client).__name__,
        "modelo_llm": OPENAI_MODEL if HAS_OPENAI_KEY else "demo-local",
        "aprovado_validacao_automatica": individual_validation["aprovado"],
        "secoes_ausentes": individual_validation["secoes_ausentes"],
        "termos_proibidos": individual_validation["termos_proibidos"],
        "arquivo_resposta": "resposta_individual.md",
    },
    {
        "tipo_interpretacao": "agregada",
        "llm_client": type(llm_client).__name__,
        "modelo_llm": OPENAI_MODEL if HAS_OPENAI_KEY else "demo-local",
        "aprovado_validacao_automatica": aggregate_validation["aprovado"],
        "secoes_ausentes": aggregate_validation["secoes_ausentes"],
        "termos_proibidos": aggregate_validation["termos_proibidos"],
        "arquivo_resposta": "resposta_agregada.md",
    },
]

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUTPUT_DIR / "auditoria_interpretacoes_llm.csv", index=False)
model_metrics.to_csv(OUTPUT_DIR / "metricas_modelos_para_llm.csv", index=False)

audit_df

#### Item 9 - Conclusao da integracao

A Parte 10 implementa uma camada de interpretacao com LLM mantendo separacao clara entre predicao estatistica e decisao clinica.

Pontos principais:

- a chave da OpenAI nao e salva no notebook;
- os prompts usam payloads estruturados para reduzir ambiguidade;
- dados identificadores sao removidos antes da chamada;
- a resposta exige secoes fixas para facilitar auditoria;
- ha validacao automatica inicial para secoes obrigatorias, cautela medica, numeros e termos proibidos;
- o fluxo esta pronto para receber os resultados finais recalculados dos modelos otimizados por Algoritmo Genetico.

Antes de qualquer uso real, as respostas devem ser revisadas por humanos e validadas em dados prospectivos.